# Industrial Anomaly Detection — Environment Setup

This notebook verifies the development environment for the industrial anomaly detection project.

It checks:

- Python and virtual environment
- Installed library versions
- PyTorch and CUDA availability
- NVIDIA GTX 1050 GPU information
- Basic GPU computation
- Project folder structure

In [1]:
from pathlib import Path
import platform
import sys

PROJECT_ROOT = Path(r"D:\PROJECTS\ANOMALY DETECTION")

print("=" * 70)
print("PROJECT AND PYTHON ENVIRONMENT")
print("=" * 70)

print(f"Project root:       {PROJECT_ROOT}")
print(f"Project exists:     {PROJECT_ROOT.exists()}")
print(f"Python version:     {platform.python_version()}")
print(f"Python executable:  {sys.executable}")
print(f"Operating system:   {platform.platform()}")

expected_python = PROJECT_ROOT / ".venv" / "Scripts" / "python.exe"

print(f"Expected interpreter: {expected_python}")
print(
    "Correct virtual environment:",
    Path(sys.executable).resolve() == expected_python.resolve(),
)

PROJECT AND PYTHON ENVIRONMENT
Project root:       D:\PROJECTS\ANOMALY DETECTION
Project exists:     True
Python version:     3.11.4
Python executable:  d:\PROJECTS\ANOMALY DETECTION\.venv\Scripts\python.exe
Operating system:   Windows-10-10.0.19045-SP0
Expected interpreter: D:\PROJECTS\ANOMALY DETECTION\.venv\Scripts\python.exe
Correct virtual environment: True


In [2]:
import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import sklearn
import skimage
import torch
import torchvision
import tqdm
import yaml

print("INSTALLED LIBRARY VERSIONS:\n")

versions = {
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
    "Matplotlib": matplotlib.__version__,
    "Pillow": PIL.__version__,
    "OpenCV": cv2.__version__,
    "Scikit-learn": sklearn.__version__,
    "Scikit-image": skimage.__version__,
    "PyTorch": torch.__version__,
    "Torchvision": torchvision.__version__,
    "TQDM": tqdm.__version__,
    "PyYAML": yaml.__version__,
}

for library, version in versions.items():
    print(f"{library:<18}: {version}")

print("\nAll core libraries imported successfully.")

INSTALLED LIBRARY VERSIONS:

NumPy             : 2.4.4
Pandas            : 3.0.3
Matplotlib        : 3.11.0
Pillow            : 12.2.0
OpenCV            : 4.13.0
Scikit-learn      : 1.9.0
Scikit-image      : 0.26.0
PyTorch           : 2.5.1+cu118
Torchvision       : 0.20.1+cu118
TQDM              : 4.68.3
PyYAML            : 6.0.3

All core libraries imported successfully.


In [3]:
print("CUDA AND GPU INFORMATION:\n")

cuda_available = torch.cuda.is_available()

print(f"PyTorch version:      {torch.__version__}")
print(f"PyTorch CUDA runtime: {torch.version.cuda}")
print(f"CUDA available:       {cuda_available}")

if cuda_available:
    device_index = torch.cuda.current_device()
    gpu_properties = torch.cuda.get_device_properties(device_index)

    print(f"GPU index:            {device_index}")
    print(f"GPU name:             {gpu_properties.name}")
    print(
        f"Total VRAM:           "
        f"{gpu_properties.total_memory / (1024 ** 3):.2f} GB"
    )
    print(
        f"Compute capability:   "
        f"{gpu_properties.major}.{gpu_properties.minor}"
    )
else:
    print("GPU was not detected. The project will use the CPU.")

CUDA AND GPU INFORMATION:

PyTorch version:      2.5.1+cu118
PyTorch CUDA runtime: 11.8
CUDA available:       True
GPU index:            0
GPU name:             NVIDIA GeForce GTX 1050
Total VRAM:           4.00 GB
Compute capability:   6.1


In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device: {DEVICE}")

Selected device: cuda


In [5]:
import time

print("GPU COMPUTATION TEST")

matrix_size = 2000

try:
    matrix_a = torch.randn(
        matrix_size,
        matrix_size,
        device=DEVICE,
    )

    matrix_b = torch.randn(
        matrix_size,
        matrix_size,
        device=DEVICE,
    )

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    matrix_result = torch.matmul(matrix_a, matrix_b)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = time.perf_counter() - start_time

    print(f"Result shape:          {matrix_result.shape}")
    print(f"Result device:         {matrix_result.device}")
    print(f"Execution time:        {elapsed_time:.4f} seconds")

    if DEVICE.type == "cuda":
        allocated_memory = torch.cuda.memory_allocated() / (1024 ** 2)
        reserved_memory = torch.cuda.memory_reserved() / (1024 ** 2)

        print(f"GPU memory allocated: {allocated_memory:.2f} MB")
        print(f"GPU memory reserved:  {reserved_memory:.2f} MB")

    print("\nGPU computation test completed successfully.")

finally:
    for variable_name in [
        "matrix_a",
        "matrix_b",
        "matrix_result",
    ]:
        if variable_name in globals():
            del globals()[variable_name]

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

GPU COMPUTATION TEST
Result shape:          torch.Size([2000, 2000])
Result device:         cuda:0
Execution time:        0.5396 seconds
GPU memory allocated: 56.12 MB
GPU memory reserved:  68.00 MB

GPU computation test completed successfully.


In [6]:
print("PROJECT FOLDER VERIFICATION")

required_directories = [
    "configs",
    "data",
    "data/raw",
    "data/processed",
    "models",
    "notebooks",
    "results",
    "results/figures",
    "results/metrics",
    "src",
    "src/data",
    "src/models",
    "src/evaluation",
    "src/visualization",
]

missing_directories = []

for relative_directory in required_directories:
    directory_path = PROJECT_ROOT / relative_directory
    exists = directory_path.exists()

    status = "FOUND" if exists else "MISSING"

    print(f"{status:<8} {relative_directory}")

    if not exists:
        missing_directories.append(relative_directory)

if missing_directories:
    print("\nMissing directories:")
    for directory in missing_directories:
        print(f"- {directory}")
else:
    print("\nAll required project directories exist.")

PROJECT FOLDER VERIFICATION
FOUND    configs
FOUND    data
FOUND    data/raw
FOUND    data/processed
FOUND    models
FOUND    notebooks
FOUND    results
FOUND    results/figures
FOUND    results/metrics
FOUND    src
FOUND    src/data
FOUND    src/models
FOUND    src/evaluation
FOUND    src/visualization

All required project directories exist.


In [7]:
def display_folder_tree(
    root_directory: Path,
    maximum_depth: int = 2,
) -> None:
    """Display the project folder tree up to a specified depth."""

    root_directory = root_directory.resolve()

    print(root_directory.name + "/")

    paths = sorted(
        path
        for path in root_directory.rglob("*")
        if ".venv" not in path.parts
        and ".git" not in path.parts
        and "__pycache__" not in path.parts
    )

    for path in paths:
        relative_path = path.relative_to(root_directory)
        depth = len(relative_path.parts)

        if depth > maximum_depth:
            continue

        indentation = "    " * depth
        suffix = "/" if path.is_dir() else ""

        print(f"{indentation}{path.name}{suffix}")


display_folder_tree(PROJECT_ROOT)

ANOMALY DETECTION/
    Anomaly Detection.ipynb
    configs/
    data/
        processed/
        raw/
    models/
    notebooks/
        01_environment_setup.ipynb
        02_dataset_exploration.ipynb
    results/
        figures/
        metrics/
    src/
        data/
        evaluation/
        models/
        visualization/


In [8]:
import json

environment_information = {
    "project_root": str(PROJECT_ROOT),
    "python_version": platform.python_version(),
    "python_executable": sys.executable,
    "operating_system": platform.platform(),
    "pytorch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_runtime": torch.version.cuda,
    "selected_device": str(DEVICE),
    "libraries": versions,
}

if torch.cuda.is_available():
    gpu_properties = torch.cuda.get_device_properties(0)

    environment_information["gpu"] = {
        "name": gpu_properties.name,
        "vram_gb": round(
            gpu_properties.total_memory / (1024 ** 3),
            2,
        ),
        "compute_capability": (
            f"{gpu_properties.major}.{gpu_properties.minor}"
        ),
    }

output_path = (
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "environment_info.json"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as output_file:
    json.dump(
        environment_information,
        output_file,
        indent=4,
    )

print(f"Environment information saved to:\n{output_path}")

Environment information saved to:
D:\PROJECTS\ANOMALY DETECTION\results\metrics\environment_info.json


In [9]:
print("ENVIRONMENT SETUP COMPLETE")

checks = {
    "Project folder exists": PROJECT_ROOT.exists(),
    "Correct Python version": platform.python_version().startswith("3.11"),
    "Virtual environment selected": ".venv" in sys.executable,
    "PyTorch installed": torch.__version__ is not None,
    "CUDA available": torch.cuda.is_available(),
    "Required folders exist": len(missing_directories) == 0,
}

for check_name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:<6} {check_name}")

all_checks_passed = all(checks.values())

print("\nOverall status:", "READY" if all_checks_passed else "CHECK REQUIRED")

ENVIRONMENT SETUP COMPLETE
PASS   Project folder exists
PASS   Correct Python version
PASS   Virtual environment selected
PASS   PyTorch installed
PASS   CUDA available
PASS   Required folders exist

Overall status: READY
